# Homework 3 — Classification

**Data Science for Health Informatics**


In [ ]:
student_name = "Chantale NzeggeMvele"
student_id = ""  # Student ID omitted from the public GitHub version
student_background = "Nurse anesthetist"


## 1. Complete EDA

This notebook continues the exploratory data analysis of the **UCI Adult Income dataset** from Homework 1. The EDA has been revised based on the feedback received for HW1. The dataset used here is the training portion of the Adult dataset, containing **32,561 observations and 14 predictor features plus the income target**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set the seed to make the results reproducible
RANDOM_SEED = 2025
np.random.seed(RANDOM_SEED)


In [ ]:
# Load the Adult training dataset
# The local file is included with this notebook for reproducibility.
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
local_file = 'adult.data'

column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
    'income'
]

try:
    df = pd.read_csv(local_file, header=None, names=column_names,
                     na_values=' ?', skipinitialspace=True)
    print('Loaded local adult.data file.')
except FileNotFoundError:
    df = pd.read_csv(url, header=None, names=column_names,
                     na_values=' ?', skipinitialspace=True)
    print('Local adult.data was not found; loaded the dataset from the UCI URL.')

df.head()


### Checking the number of observations

One of the feedback points from HW1 concerned the number of observations remaining after preprocessing. I therefore check the number of rows before and after handling missing values.


In [ ]:
df.shape

### Data preprocessing and feature selection

The dataset contains numerical and categorical variables. I first inspect the structure and missing values, then handle missing values before creating the cleaned dataset.


In [ ]:
# Replace missing-value markers with NaN and then remove incomplete rows.
df = df.replace('?', pd.NA)
df_cleaned = df.dropna().copy()

print('Rows before cleaning:', len(df))
print('Rows after cleaning:', len(df_cleaned))
print('Rows removed:', len(df) - len(df_cleaned))


### Data inspection

The following checks verify the number of observations, data types, and remaining missing values.


In [ ]:
df_cleaned.info()
print('\nMissing values after cleaning:')
print(df_cleaned.isnull().sum())


### Missing values

Some entries in the original dataset are represented by `?`. These values are treated as missing before rows containing missing values are removed.


In [ ]:
# Verify that no missing-value markers remain
print('Remaining missing values:', df_cleaned.isna().sum().sum())


### Rename columns for clarity


The `fnlwgt` column is renamed to `final_weight`, and `education_num` is renamed to `education_level` to make the variables easier to interpret.


In [ ]:
df_cleaned = df_cleaned.rename(columns={
    'fnlwgt': 'final_weight',
    'education_num': 'education_level'
})
df_cleaned.shape

In [ ]:
# Convert appropriate variables to categorical data types.
categorical_cols = [
    'workclass', 'education', 'marital_status', 'occupation',
    'relationship', 'race', 'sex', 'native_country', 'income'
]

for col in categorical_cols:
    df_cleaned[col] = df_cleaned[col].astype('category')

df_cleaned.dtypes


### Renaming columns

The renamed columns are used consistently in the remaining analysis.


### Handling missing values

After replacing missing-value markers and removing incomplete rows, the cleaned dataset is used for the numerical analyses below.


### Number of observations after cleaning


The HW1 feedback raised a question about missing observations. In this version, the number of removed observations is calculated directly from the data rather than assuming a specific number.


In [ ]:
print('Original rows:', len(df))
print('Cleaned rows:', len(df_cleaned))
print('Rows removed:', len(df) - len(df_cleaned))
df_cleaned.head()


## 2. Univariate data analysis

### Q1. What is the average age of individuals in the dataset?


In [ ]:
average_age = df_cleaned['age'].mean()
print('The average age of individuals in the dataset is:', round(average_age, 2))

**Reflection:** The average age gives a general description of the population. However, it does not show how ages are distributed, so it should be considered together with other descriptive measures and visualizations.


### Q2. What is the distribution of education levels in the dataset?


In [ ]:
education_counts = df_cleaned['education'].value_counts()
print(education_counts)

**Reflection:** High school graduates represent the largest education category, while Preschool is one of the smallest categories. This shows that education is distributed unevenly across the dataset.


## 3. Bivariate data analysis

### Q1. What is the average capital gain by education level?


In [ ]:
df_cleaned.groupby('education', observed=True)['capital_gain'].mean().sort_values(ascending=False)

**Reflection:** Average capital gain differs between education categories. This describes an association in the dataset, but it does not show that education itself causes differences in capital gain because other factors may also contribute.


### Q2. What is the distribution of income levels by marital status?


In [ ]:
df_cleaned.groupby('marital_status', observed=True)['income'].value_counts(normalize=True).unstack()

**Reflection:** The income proportions differ across marital-status categories. In particular, some married categories have a higher proportion of incomes above $50K. This is an association, not evidence that marital status causes higher income.


## 4. Analytical questions using visualization


### Q1. What is the distribution of hours worked per week across different educational levels?


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='education_level', y='hours_per_week', data=df_cleaned)
plt.title('Hours Worked per Week by Education Level')
plt.xlabel('Education Level (numeric code)')
plt.ylabel('Hours per Week')
plt.show()

**Reflection:** The boxplots show that weekly working hours are broadly similar across education levels, although the distributions and outliers vary. The differences are not large enough to conclude that education strongly determines working hours.


### Q2. What is the age distribution by income level?


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='income', y='age', data=df_cleaned)
plt.title('Age Distribution by Income Level')
plt.xlabel('Income')
plt.ylabel('Age')
plt.show()

**Reflection:** The boxplot indicates that the age distributions differ between the two income groups. The higher-income group tends to have a higher median age, although there is substantial overlap between the groups.


### Q3. Is there a relationship between education level and income proportion?


In [ ]:
edu_income = pd.crosstab(
    df_cleaned['education'],
    df_cleaned['income'],
    normalize='index'
)

edu_income.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Income Proportion by Education Level')
plt.xlabel('Education')
plt.ylabel('Proportion')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Reflection:** The visualization shows a general increase in the proportion of higher-income observations across several higher education categories. However, income is not determined by education alone.


### Q4. Is there any correlation between age and the number of hours worked per week?


In [ ]:
corr_age_hours = df_cleaned[['age', 'hours_per_week']].corr(method='pearson')
print('Pearson correlation between age and hours worked per week:')
print(corr_age_hours)

plt.figure(figsize=(8, 6))
sns.regplot(data=df_cleaned, x='age', y='hours_per_week', scatter_kws={'alpha': 0.2})
plt.title('Age and Hours Worked per Week')
plt.show()

**Reflection:** The Pearson correlation is weak and positive, indicating only a small linear association between age and weekly working hours. Correlation does not indicate that age causes changes in working hours.


## 5. Machine learning classification

The classification task is to predict whether an individual's annual income is above $50K using selected demographic and work-related variables.


### 5.1 Target variable and input features

**Target:** `income`

- `0` = `<=50K`
- `1` = `>50K`

**Input features:**
- `education_level_encoded` — an ordinal representation of education attainment
- `hours_per_week` — number of hours worked per week
- `age` — age of the individual

The prediction task is: **Can we predict whether an individual earns more than $50K per year from age, education level, and weekly working hours?**


### Reflection

I chose income as the target because the Adult dataset provides a natural binary outcome after grouping the two income categories. Education, age, and weekly working hours were selected because they describe educational attainment, a person's stage of working life, and work intensity. Education may be associated with access to different occupations, while age can reflect accumulated work experience. Weekly working hours may also be related to earnings. The target is binarized as 0 for income at or below $50K and 1 for income above $50K. The model will therefore predict the probability class of higher annual income from these three input features.


In [ ]:
# Create the binary target variable
df_cleaned['income_binary'] = (
    df_cleaned['income'].astype(str).str.strip().eq('>50K').astype(int)
)

print(df_cleaned['income_binary'].value_counts().sort_index())


### 5.2 Preparing the categorical feature

Education has an ordered meaning, so an ordinal representation is used. The categories are grouped into three levels to simplify the feature while preserving an increasing order of educational attainment:

- `0` = low: Preschool through 12th grade
- `1` = medium: HS-grad through associate-level education
- `2` = high: Bachelor through Doctorate


In [ ]:
def map_edu_level(edu):
    if edu in ['Preschool', '1st-4th', '5th-6th', '7th-8th', '9th', '10th', '11th', '12th']:
        return 'low'
    elif edu in ['HS-grad', 'Some-college', 'Assoc-acdm', 'Assoc-voc']:
        return 'medium'
    else:
        return 'high'

df_cleaned['education_level'] = df_cleaned['education'].apply(map_edu_level)
edu_order = ['low', 'medium', 'high']
df_cleaned['education_level'] = pd.Categorical(
    df_cleaned['education_level'], categories=edu_order, ordered=True
)
df_cleaned['education_level_encoded'] = df_cleaned['education_level'].cat.codes


### Checking the numerical representation


In [ ]:
classification_columns = [
    'education_level_encoded', 'hours_per_week', 'age', 'income_binary'
]

print(df_cleaned[classification_columns].dtypes)
print('All selected classification variables are numerical:',
      all(pd.api.types.is_numeric_dtype(df_cleaned[col]) for col in classification_columns))


In [ ]:
# Convert the selected variables into NumPy arrays as expected by scikit-learn classifiers.
X = df_cleaned[['education_level_encoded', 'hours_per_week', 'age']].to_numpy()
y = df_cleaned['income_binary'].to_numpy()

print('X shape:', X.shape)
print('y shape:', y.shape)


### 5.3 Class imbalance


In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(y, bins=[-0.5, 0.5, 1.5], edgecolor='black', rwidth=0.8)
plt.title('Distribution of Target Variable (Income)')
plt.xlabel('Income Class (0 = <=50K, 1 = >50K)')
plt.ylabel('Count')
plt.xticks([0, 1])
plt.show()

class_counts = pd.Series(y).value_counts().sort_index()
print(class_counts)
print('Class proportions:')
print((class_counts / len(y)).round(3))


**Reflection:** Class imbalance occurs when target classes are represented unequally. In this dataset, the `<=50K` class is the majority class. A classifier could therefore obtain reasonable accuracy by favoring the majority class, while performing less well on the minority `>50K` class. Precision, recall, and F1-score are therefore important alongside accuracy.


### 5.4 Separate X and y


In [ ]:
# X contains the three predictors and y contains only the binary target.
X = df_cleaned[['education_level_encoded', 'hours_per_week', 'age']].to_numpy()
y = df_cleaned['income_binary'].to_numpy()

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Unique target values:', np.unique(y))


### 5.5 Stratified train-test partitioning

Stratified partitioning keeps approximately the same proportion of the two income classes in both the training and test sets. This is useful here because the target classes are imbalanced. An 80/20 split is used, with the test set kept aside until the final evaluation.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)

print('Training set:', X_train.shape, y_train.shape)
print('Test set:', X_test.shape, y_test.shape)
print('Training class proportions:', np.bincount(y_train) / len(y_train))
print('Test class proportions:', np.bincount(y_test) / len(y_test))


**Reflection:** Stratified splitting preserves the proportion of each target class in both subsets. This is particularly useful for the Adult dataset because the `>50K` group is smaller. It reduces the risk that a random split produces an unrepresentative test set.


## 6. Experimental evaluation of classifiers

The evaluation follows the classification lab approach. Ten classifier variants are compared using cross-validation **only on the training set**. The held-out test set is used once for the final evaluation of the selected model.


### 6.1 Ten classifier variants

Two variants are used for each of Decision Tree, Random Forest, K-Nearest Neighbors, and Support Vector Machine. Gaussian Naive Bayes is used as the additional classifier, with two values of `var_smoothing`.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

classifiers = {
    'DT_depth3': DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
    'DT_depth7': DecisionTreeClassifier(max_depth=7, random_state=RANDOM_SEED),

    'RF_50trees': RandomForestClassifier(n_estimators=50, random_state=RANDOM_SEED),
    'RF_200trees': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_SEED),

    'KNN_k3': Pipeline([
        ('scaler', MinMaxScaler()),
        ('classifier', KNeighborsClassifier(n_neighbors=3))
    ]),
    'KNN_k7': Pipeline([
        ('scaler', MinMaxScaler()),
        ('classifier', KNeighborsClassifier(n_neighbors=7))
    ]),

    'SVM_rbf': Pipeline([
        ('scaler', MinMaxScaler()),
        ('classifier', SVC(kernel='rbf', C=1))
    ]),
    'SVM_linear': Pipeline([
        ('scaler', MinMaxScaler()),
        ('classifier', SVC(kernel='linear', C=0.5))
    ]),

    'NB_default': GaussianNB(),
    'NB_smoothing_1e-8': GaussianNB(var_smoothing=1e-8)
}

print('Number of classifier variants:', len(classifiers))
print(list(classifiers.keys()))


### 6.2 Cross-validation, performance and computational efficiency

Five-fold stratified cross-validation is used on the training set. The evaluation records accuracy, precision, recall, F1-score, mean fitting time, and mean prediction/score time. F1-score is especially useful here because the target classes are imbalanced.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

results = []

for name, model in classifiers.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=1
    )

    results.append({
        'Classifier': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),
        'Train Time (s)': scores['fit_time'].mean(),
        'Prediction Time (s)': scores['score_time'].mean()
    })

results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)

print('Cross-validation results:')
display(results_df.round(4))


### 6.3 Comparing the classifiers


In [ ]:
# Display the cross-validation results, sorted by F1-score.
results_df.round(4)


### 6.4 Selecting and evaluating the best model

The model with the highest mean cross-validation F1-score is selected. It is then fitted on the complete training set and evaluated on the untouched test set.


In [ ]:
best_model_name = results_df.loc[results_df['F1-Score'].idxmax(), 'Classifier']
best_model = classifiers[best_model_name]

print('Best model based on mean cross-validation F1-score:', best_model_name)

best_model.fit(X_train, y_train)
y_test_pred = best_model.predict(X_test)

final_accuracy = accuracy_score(y_test, y_test_pred)
final_precision = precision_score(y_test, y_test_pred, zero_division=0)
final_recall = recall_score(y_test, y_test_pred, zero_division=0)
final_f1 = f1_score(y_test, y_test_pred, zero_division=0)

print('Final test-set performance:')
print(f'Accuracy:  {final_accuracy:.4f}')
print(f'Precision: {final_precision:.4f}')
print(f'Recall:    {final_recall:.4f}')
print(f'F1-score:  {final_f1:.4f}')


### Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cbar=False)
plt.title(f'Confusion Matrix — {best_model_name}')
plt.xlabel('Predicted class')
plt.ylabel('Actual class')
plt.xticks([0.5, 1.5], ['<=50K', '>50K'])
plt.yticks([0.5, 1.5], ['<=50K', '>50K'], rotation=0)
plt.show()

print(classification_report(y_test, y_test_pred, target_names=['<=50K', '>50K'], zero_division=0))


## 7. Analytical questions on the evaluation results

### Q1. Which classifier provides the best predictive performance according to the cross-validation F1-score?

**Reflection:** The classifier with the highest mean cross-validation F1-score provides the strongest balance between precision and recall for the minority income class. This makes F1-score useful alongside accuracy for this imbalanced dataset.


### Q2. How do the training and prediction times differ between the classifiers?

**Reflection:** Simpler models such as Naive Bayes and Decision Trees generally require less computation, while larger forests and SVM models can require more time. This creates a performance-efficiency trade-off.


### Q3. How do different hyperparameters affect classifier performance?

**Reflection:** Changing parameters such as tree depth, number of trees, neighbors, or SVM kernel changes model complexity. The results show that tuning can improve performance, but more complex settings do not always perform better.


## 8. Final reflection: deployment

The best model in this experiment is selected using cross-validation F1-score and then tested on unseen data. I would not deploy it directly in a real-life setting because the model uses a limited set of features and the Adult dataset is not a clinical dataset. External validation, fairness assessment, monitoring, and a clearly defined use case would be needed before deployment.
